In [1]:
import pandas as pd
import re
from sqlalchemy import create_engine
import urllib

In [2]:
# Configurar cadena de conexión
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=comerssiamirror.eastus2.cloudapp.azure.com,38693;"
    "DATABASE=PROVENZAL;"
    "UID=provenzal;"
    "PWD=PE@4:BRy/<1VWkp;"
)

# Crear el motor
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [3]:
query = """
SELECT DISTINCT c.CLICodigo,
       c.CLINombres,
       c.CLIApellidos,
       c.CLIEmailPrincipal,
       c.CLICelular,
       C.CLITipoIdentificacion
FROM Clientes c
INNER JOIN Ventas_MoviNova v ON c.CLICodigo = v.Cliente
--WHERE v.Fecha >= '2025-10-01'
"""

# Ejecutar y cargar en DataFrame
df = pd.read_sql(query, engine)


# Obtener los CLICodigos ya existentes en la tabla SQL
# clientes_existentes = pd.read_sql("SELECT CLICodigo FROM Contacto_Clientes", con=engine)

In [4]:
#Limpiar Celular
def limpiar_celular(cel):
    if not cel:
        return ""
    cel = str(cel).strip().replace(" ", "").replace("-", "").replace("(", "").replace(")", "")
    if cel.startswith("+57"):
        cel = cel[3:]
    elif cel.startswith("57") and len(cel) > 10:
        cel = cel[2:]
    return cel

df['CLICelular'] = df['CLICelular'].apply(limpiar_celular)

#Validar celular
def es_celular_valido(cel):
    return bool(re.fullmatch(r"3\d{9}", str(cel)))

df['CelularValido'] = df['CLICelular'].apply(es_celular_valido)

#Limpiar y validar email
df['CLIEmailPrincipal'] = df['CLIEmailPrincipal'].apply(lambda x: str(x).strip())

def es_email_valido(email):
    email = str(email).strip()

    # Si está vacío
    if not email:
        return False
    
    # Palabras prohibidas
    palabras_no_permitidas = ["negad", "pendi"]
    if any(palabra in email for palabra in palabras_no_permitidas):
        return False
    
    # Validación de formato
    return bool(re.fullmatch(r'^[\w\.-]+@[\w\.-]+\.\w+$', email))

df['EmailValido'] = df['CLIEmailPrincipal'].apply(es_email_valido)

In [5]:
# # Detectar duplicados
# cel_duplicados = df['CLICelular'].duplicated(keep=False)
# email_duplicados = df['CLIEmailPrincipal'].duplicated(keep=False)

# # Marcar celulares duplicados como no válidos
# df.loc[cel_duplicados, 'CelularValido'] = False

# # Marcar correos duplicados como no válidos
# df.loc[email_duplicados, 'EmailValido'] = False

In [6]:
#Agregar columna contacto
def determinar_contacto(row):
    if row['CelularValido'] and row['EmailValido']:
        return "Cel + Email"
    elif row['CelularValido']:
        return "Cel"
    elif row['EmailValido']:
        return "Email"
    else:
        return "Falso"

df['Contacto'] = df.apply(determinar_contacto, axis=1)

In [7]:
# # # Filtrar para dejar solo nuevos clientes
# df_nuevos = df[~df["CLICodigo"].isin(clientes_existentes["CLICodigo"])]

In [8]:
# # Insertar solo los nuevos en SQL
# if not df_nuevos.empty:
#     df_nuevos.to_sql("Contacto_Clientes", con=engine, if_exists="append", index=False)
#     print(f"{len(df_nuevos)} nuevos clientes insertados.")
# else:
#     print("No hay nuevos clientes por insertar.")

In [9]:
# Exportar todos los registros con validaciones incluidas
# df.to_excel("clientes_con_validacion.xlsx", index=False)

In [10]:
df.to_sql("Contacto_Clientes", engine, if_exists="replace", index=False)

133